# KKR Auction Scout: uploaded sample analysis

Run the ingestion and scoring pipeline first. This notebook reads computed findings from the database. It contains no copied statistics or sample player rows.

The uploaded sample is incomplete coverage of each competition. The exploratory intervals use an innings-level t interval and treat innings as independent. They do not account for how matches were selected for upload.

In [ ]:
import json
import sys
from pathlib import Path
from sqlalchemy import text

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))
from kkr_scout.database import get_engine
engine = get_engine(project_root)
with engine.connect() as connection:
    findings = json.loads(connection.execute(text("SELECT value FROM pipeline_meta WHERE key='eda'")).scalar_one())
print(findings['method'])
for finding in findings['findings']:
    print(finding['insight'])

## Counting audit

Ball counts and run numerators below are queried from regulation deliveries. Super overs are excluded. Wides and no-balls do not enter the balls-faced or legal-balls denominators.

In [ ]:
with engine.connect() as connection:
    audit = connection.execute(text('''SELECT competition, COUNT(*) delivery_events,
        SUM(batter_ball) balls_faced, SUM(is_legal) legal_balls,
        SUM(runs_bat) bat_runs, SUM(bowler_runs) bowler_runs
        FROM regulation_deliveries GROUP BY competition''')).mappings().all()
for row in audit:
    print(dict(row))

## League bridges and measured holdout

The auction back-test requires missing auction and uncapped-eligibility records. A separate last-season comparison reports observed results from players who meet the documented sample limits. Training ratings use only records before the holdout year.

In [ ]:
with engine.connect() as connection:
    model = json.loads(connection.execute(text("SELECT value FROM pipeline_meta WHERE key='model_summary'")).scalar_one())
print(json.dumps(model['league_adjustment'], indent=2))
print(json.dumps(model['backtest'], indent=2))

## Venue and season par totals

Par totals use first innings from completed full-length matches, excluding method-marked and no-result matches. Small venue-season samples remain marked as insufficient.

In [ ]:
for par in model['venue_par']:
    if par['status'] == 'Available':
        print(par)